In [ ]:
import os
import pandas as pd
import pybedtools
import matplotlib.pyplot as plt
import numpy as np

base_dir = "/Path/to/Project/"

prdm9_13mer_dir = os.path.join(base_dir, "final_analysis/data/prdm9/prdm9b_Human_1_to_6.bed")
prdm9_B_dir = os.path.join(base_dir, "final_analysis/data/prdm9/prdm9b_Human7.bed")


In [ ]:
# ------------------------
# Load motifs
# ------------------------
fimo_cols = [
    "chrom", "Start", "End", "motif_id", 
    "strand", "score", "pvalue", "qvalue", "motif_seq"
]

# 13-mer PRDM9 A motif
prdm13mer = pd.read_csv(
    prdm9_13mer_dir,
    sep="\t",
    comment="#",
    header=None,
    names=fimo_cols
)[["chrom", "Start", "End", "strand"]]
prdm13mer = prdm13mer[~prdm13mer["chrom"].isin(["chrX", "chrY", "chrM"])]

# PRDM9 B-specific motif (Human7)
prdmB = pd.read_csv(
    prdm9_B_dir,
    sep="\t",
    comment="#",
    header=None,
    names=fimo_cols
)[["chrom", "Start", "End", "strand"]]
prdmB = prdmB[~prdmB["chrom"].isin(["chrX", "chrY", "chrM"])]


motifs = {"PRDM-13mer": prdm13mer, "PRDM-B": prdmB}

In [ ]:
def clip_map_to_allowed_regions(df_map: pd.DataFrame,
                               allowed_df: pd.DataFrame,
                               chrom: str,
                               chrom_col_allowed: str = "chr",
                               start_col_allowed: str = "Start",
                               end_col_allowed: str = "End") -> pd.DataFrame:

    # Allowed intervals for this chromosome
    allowed = allowed_df[allowed_df[chrom_col_allowed].astype(str) == str(chrom)][
        [start_col_allowed, end_col_allowed]
    ].copy()

    if allowed.empty or df_map.empty:
        return df_map.iloc[0:0].copy()

    # sort for safety
    allowed = allowed.sort_values([start_col_allowed, end_col_allowed]).to_numpy()
    df_map = df_map.sort_values(["Start", "End"]).reset_index(drop=True)

    out_rows = []

    # Two-pointer sweep (fast enough; map windows are usually not huge)
    j = 0
    for s, e, r in df_map[["Start", "End", "Rec.Rate"]].to_numpy():
        if e <= s:
            continue

        # advance allowed pointer until it might overlap
        while j < len(allowed) and allowed[j][1] <= s:
            j += 1

        k = j
        # collect all overlaps with allowed intervals
        while k < len(allowed) and allowed[k][0] < e:
            a_s, a_e = allowed[k]
            ov_s = max(s, a_s)
            ov_e = min(e, a_e)
            if ov_s < ov_e:
                out_rows.append((ov_s, ov_e, r))
            if a_e >= e:
                break
            k += 1

    if not out_rows:
        return df_map.iloc[0:0].copy()

    df_out = pd.DataFrame(out_rows, columns=["Start", "End", "Rec.Rate"])
    df_out = df_out.sort_values(["Start", "End"]).reset_index(drop=True)
    return df_out



In [ ]:
pan_available = os.path.join(base_dir, "CHM13v2.telo_cent.complement.bed")
pan_available_region = pd.read_csv(
	pan_available, sep="\t", header=None, names=["Chrom", "Start", "End"]
)
# remove chrX and chrY in Chrom
pan_available_region = pan_available_region[~pan_available_region["Chrom"].isin(["chrX", "chrY"])]
pan_available_region ["chr"] = pan_available_region ["Chrom"].str.replace("chr", "")
pan_available_region

In [ ]:
def filter_prdm_to_allowed_regions(
    prdm_df: pd.DataFrame,
    allowed_df: pd.DataFrame,
    chrom_col_prdm: str = "chrom",
    start_col_prdm: str = "Start",
    end_col_prdm: str = "End",
    chrom_col_allowed: str = "Chrom",
    start_col_allowed: str = "Start",
    end_col_allowed: str = "End",) -> pd.DataFrame:
    """
    Keep PRDMb records that overlap any allowed region on the same chromosome.
    No clipping, no splitting.
    """

    out = []

    # process chromosome by chromosome (fast + clean)
    for chrom, pr_chr in prdm_df.groupby(chrom_col_prdm):
        allowed_chr = allowed_df[
            allowed_df[chrom_col_allowed] == chrom
        ][[start_col_allowed, end_col_allowed]]

        if allowed_chr.empty:
            continue

        allowed = allowed_chr.sort_values(
            [start_col_allowed, end_col_allowed]
        ).to_numpy()

        pr = pr_chr[[start_col_prdm, end_col_prdm]].to_numpy()

        j = 0
        for s, e in pr:
            # advance allowed pointer
            while j < len(allowed) and allowed[j][1] <= s:
                j += 1

            k = j
            keep = False
            while k < len(allowed) and allowed[k][0] < e:
                if e > allowed[k][0] and s < allowed[k][1]:
                    keep = True
                    break
                k += 1

            if keep:
                out.append((chrom, s, e))

    if not out:
        return prdm_df.iloc[0:0].copy()

    df_out = pd.DataFrame(out, columns=[chrom_col_prdm,
                                        start_col_prdm,
                                        end_col_prdm])
    # merge back any extra columns if needed
    df_out = df_out.merge(
        prdm_df,
        on=[chrom_col_prdm, start_col_prdm, end_col_prdm],
        how="left"
    )

    return df_out.reset_index(drop=True)


prdmB = filter_prdm_to_allowed_regions(prdmB, pan_available_region)
prdm13mer = filter_prdm_to_allowed_regions(prdm13mer, pan_available_region)

print(len(prdmB))
print(len(prdm13mer))

In [ ]:
cha_ngs = os.path.join(base_dir, "final_analysis/data/CHA/ngs/bp35w60")
cha_pan = os.path.join(base_dir, "final_analysis/data/CHA/pan/bp35w60")
ceu_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CEU")
chb_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHB")
chs_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHS")
fin_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/FIN")
yri_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/YRI")
jpt_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/JPT")

file_patterns = {
    cha_ngs: "CHA_recombmap_chr{chrom}_bp35w60",
    cha_pan: "CHA_recombmap_chr{chrom}_bp35w60",
	ceu_unmasked_dir: "CEU_chr{chrom}_no_mask.txt",
	chb_unmasked_dir: "CHB_chr{chrom}_no_mask.txt",
	chs_unmasked_dir: "CHS_chr{chrom}_no_mask.txt",
	fin_unmasked_dir: "FIN_chr{chrom}_no_mask.txt",
	yri_unmasked_dir: "YRI_chr{chrom}_no_mask.txt",
	jpt_unmasked_dir: "JPT_chr{chrom}_no_mask.txt",

}


In [ ]:
def load_map(directory, chrom):
    if directory not in file_patterns:
        raise ValueError(f"Unknown directory: {directory}")
    file_name = file_patterns[directory].format(chrom=chrom)
    file_path = os.path.join(directory, file_name)
    
    if directory in [cha_pan, cha_ngs]:
        df = pd.read_csv(file_path, sep="\t", header=None, names=["Start", "End", "Rec.Rate"])
    else:
        df = pd.read_csv(file_path)
    return df


In [ ]:
import os
import numpy as np
import pandas as pd


def compute_recombination_profile_fast(
    map_dir,                      
    prdm_df,
    pan_available_region,
    window=8000,
    bin_size=200,
    output_file="recomb_rate_output.csv"
):


    if map_dir not in file_patterns:
        raise ValueError("map_dir not found in file_patterns")

    # ------------------------------------
    # 1. Load recombination maps
    # ------------------------------------
    pop_rate = {}
    chroms = [str(i) for i in range(1, 23)]

    for chrom in chroms:

        df = load_map(map_dir, chrom)

        df = clip_map_to_allowed_regions(
            df_map=df,
            allowed_df=pan_available_region,
            chrom=str(chrom),
            chrom_col_allowed="chr",
            start_col_allowed="Start",
            end_col_allowed="End"
        )

        df = df.sort_values("Start")

        pop_rate[chrom] = {
            "starts": df["Start"].values,
            "ends": df["End"].values,
            "rates": df["Rec.Rate"].values
        }

    # ------------------------------------
    # 2. Prepare bins
    # ------------------------------------
    bins = np.arange(0, window + bin_size, bin_size)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    n_bins = len(bin_centers)

    all_bin_values = [[] for _ in range(n_bins)]

    prdm_df = prdm_df.copy()
    prdm_df["chrom"] = prdm_df["chrom"].str.replace("chr", "")

    # ------------------------------------
    # 3. Loop through motifs
    # ------------------------------------
    for chrom in prdm_df["chrom"].unique():

        if chrom not in pop_rate:
            continue

        print(f"Processing chromosome {chrom}")

        motifs_chr = prdm_df[prdm_df["chrom"] == chrom]
        map_data = pop_rate[chrom]

        map_starts = map_data["starts"]
        map_ends = map_data["ends"]
        map_rates = map_data["rates"]

        for _, row in motifs_chr.iterrows():

            center = (row["Start"] + row["End"]) / 2

            for i, dist in enumerate(bin_centers):

                for pos in (center + dist, center - dist):

                    idx = np.searchsorted(map_starts, pos) - 1

                    if 0 <= idx < len(map_rates):
                        if map_starts[idx] <= pos < map_ends[idx]:
                            all_bin_values[i].append(map_rates[idx])

    # ------------------------------------
    # 4. Aggregate across observations
    # ------------------------------------
    mean_profile = np.zeros(n_bins)
    ci95_lower = np.zeros(n_bins)
    ci95_upper = np.zeros(n_bins)

    for i in range(n_bins):

        values = np.array(all_bin_values[i])

        if len(values) == 0:
            mean_profile[i] = np.nan
            ci95_lower[i] = np.nan
            ci95_upper[i] = np.nan
            continue

        mean = values.mean()
        std = values.std(ddof=1) if len(values) > 1 else 0
        sem = std / np.sqrt(len(values))
        ci95 = 1.96 * sem

        mean_profile[i] = mean
        ci95_lower[i] = mean - ci95
        ci95_upper[i] = mean + ci95

    df_plot = pd.DataFrame({
        "distance_bp": bin_centers,
        "mean_recomb_rate": mean_profile,
        "ci95_lower": ci95_lower,
        "ci95_upper": ci95_upper
    })

    df_plot.to_csv(output_file, index=False)

    print(f"Saved to {output_file}")

    return df_plot

In [ ]:
df_ceu_prdmb = compute_recombination_profile_fast(
    map_dir=ceu_unmasked_dir,
    prdm_df=prdmB,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdmb_CEU.csv"
)
df_ceu_prdm_consensus = compute_recombination_profile_fast(
    map_dir=ceu_unmasked_dir,
    prdm_df=prdm13mer,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdm_consensus_CEU.csv"
)

In [ ]:
df_cha_pan_prdmb = compute_recombination_profile_fast(
    map_dir=cha_pan,
    prdm_df=prdmB,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdmb_CHA_pan.csv"
)
df_cha_pan_prdm_consensus = compute_recombination_profile_fast(
    map_dir=cha_pan,
    prdm_df=prdm13mer,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdm_consensus_CHA_pan.csv"
)

In [ ]:
df_chb_prdmb = compute_recombination_profile_fast(
    map_dir=chb_unmasked_dir,
    prdm_df=prdmB,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdmb_CHB.csv"
)
df_chb_prdm_consensus = compute_recombination_profile_fast(
    map_dir=chb_unmasked_dir,
    prdm_df=prdm13mer,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdm_consensus_CHB.csv"
)

In [ ]:
df_fin_prdmb = compute_recombination_profile_fast(
    map_dir=fin_unmasked_dir,
    prdm_df=prdmB,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdmb_FIN.csv"
)
df_fin_prdm_consensus = compute_recombination_profile_fast(
    map_dir=fin_unmasked_dir,
    prdm_df=prdm13mer,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdm_consensus_FIN.csv"
)

In [ ]:
df_yri_prdmb = compute_recombination_profile_fast(
    map_dir=yri_unmasked_dir,
    prdm_df=prdmB,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdmb_YRI.csv"
)
df_yri_prdm_consensus = compute_recombination_profile_fast(
    map_dir=yri_unmasked_dir,
    prdm_df=prdm13mer,
    pan_available_region=pan_available_region,
    window=8000,
    bin_size=50,
    output_file="recomb_rate_prdm_consensus_YRI.csv"
)

In [ ]:
# normalize by population average recombination rate

In [ ]:

def compute_all_population_genomewide_means(pan_available_region):
    """
    Compute genome-wide length-weighted mean recombination rate
    for all populations defined in file_patterns.
    Returns a single DataFrame.
    """

    results = []
    chroms = [str(i) for i in range(1, 23)]

    for map_dir, pattern in file_patterns.items():

        total_weighted_rate = 0.0
        total_length = 0.0

        for chrom in chroms:

            df = load_map(map_dir, chrom)

            df = clip_map_to_allowed_regions(
                df_map=df,
                allowed_df=pan_available_region,
                chrom=str(chrom),
                chrom_col_allowed="chr",
                start_col_allowed="Start",
                end_col_allowed="End"
            )

            df = df.sort_values("Start")

            lengths = df["End"] - df["Start"]
            rates = df["Rec.Rate"]

            total_weighted_rate += np.sum(rates * lengths)
            total_length += np.sum(lengths)

        genomewide_mean = total_weighted_rate / total_length

        # Extract readable population name from filename pattern
        pop_name = pattern.split("_")[0]

        results.append({
            "population": pop_name,
            "map_dir": map_dir,
            "genomewide_weighted_mean_rate": genomewide_mean,
            "total_length_bp": total_length
        })

    return pd.DataFrame(results)

In [ ]:
df_population_means = compute_all_population_genomewide_means(
    pan_available_region=pan_available_region
)

print(df_population_means)

In [ ]:
plt.figure(figsize=(8,6))

# ----------------------
# CHB
# ----------------------
plt.plot(df_chb_prdm_consensus["distance_bp"],
         df_chb_prdm_consensus["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="CHB"]["genomewide_weighted_mean_rate"].values[0]),
         color="orange",
         label="CHB consensus")

plt.plot(df_chb_prdmb["distance_bp"],
         df_chb_prdmb["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="CHB"]["genomewide_weighted_mean_rate"].values[0]),
         color="orange",
         linestyle="dotted",
         label="CHB prdmb")


# ----------------------
# CHA PAN
# ----------------------
plt.plot(df_cha_pan_prdm_consensus["distance_bp"],
         df_cha_pan_prdm_consensus["mean_recomb_rate"]/(df_population_means[df_population_means["population"].str.contains("CHA")]["genomewide_weighted_mean_rate"].values[0]),
         color="brown",
         label="CHA pan consensus")

plt.plot(df_cha_pan_prdmb["distance_bp"],
         df_cha_pan_prdmb["mean_recomb_rate"]/(df_population_means[df_population_means["population"].str.contains("CHA")]["genomewide_weighted_mean_rate"].values[0]),
         color="brown",
		 linestyle="dotted",
         label="CHA pan prdmb")

# ----------------------
# CEU
# ----------------------
plt.plot(df_ceu_prdm_consensus["distance_bp"],
         df_ceu_prdm_consensus["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="CEU"]["genomewide_weighted_mean_rate"].values[0]),
         color="steelblue",
         label="CEU consensus")

plt.plot(df_ceu_prdmb["distance_bp"],
         df_ceu_prdmb["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="CEU"]["genomewide_weighted_mean_rate"].values[0]),
         color="steelblue",
		 linestyle="dotted",
         label="CEU prdmb")



# ----------------------
# YRI
# ----------------------
plt.plot(df_yri_prdm_consensus["distance_bp"],
         df_yri_prdm_consensus["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="YRI"]["genomewide_weighted_mean_rate"].values[0]),
         color="mediumseagreen",
         label="YRI consensus")

plt.plot(df_yri_prdmb["distance_bp"],
         df_yri_prdmb["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="YRI"]["genomewide_weighted_mean_rate"].values[0]),
         color="mediumseagreen",
		 linestyle="dotted",
         label="YRI prdmb")



# ----------------------
# FIN
# ----------------------
plt.plot(df_fin_prdm_consensus["distance_bp"],
         df_fin_prdm_consensus["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="FIN"]["genomewide_weighted_mean_rate"].values[0]),
         color="purple",
         label="FIN consensus")

plt.plot(df_fin_prdmb["distance_bp"],
         df_fin_prdmb["mean_recomb_rate"]/(df_population_means[df_population_means["population"]=="FIN"]["genomewide_weighted_mean_rate"].values[0]),
         color="purple",
		 # make it a dotted line
         linestyle="dotted",
         label="FIN prdmb")
# ----------------------
plt.xlabel("Distance from PRDM-B binding motif center (bp)")
plt.ylabel("Normalized Recombination rate by population genome-wide rate")
plt.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()